# 🎨 JsonRenderer — Demo Notebook

This notebook demonstrates the two core skills of **JsonRenderer**:
1. **`render_json_to_html`** — Convert JSON to a self-contained, styled HTML document
2. **`render_json_to_markdown`** — Convert JSON to clean Markdown

Both handle unicode, dollar signs, XSS payloads, embedded markdown, and deeply nested structures.

## 1. Setup & Imports

In [ ]:
import json
from pathlib import Path
from IPython.display import HTML, Markdown, display

from json_renderer import render_json_to_html, render_json_to_markdown

print("✅ JsonRenderer imported successfully!")

## 2. Load the Complex Test JSON

This adversarial JSON file contains every edge case we need to handle:
- Unicode (emoji, CJK, RTL, combining diacritics, ZWJ sequences)
- Dollar signs (single, double, triple, at boundaries)
- HTML injection payloads (`<script>` tags)
- Embedded Markdown in values
- Deeply nested structures (5 levels deep)
- Empty values (string, array, object)
- Special characters in keys (pipes, backslashes, ampersands)

In [ ]:
fixture_path = Path("../tests/fixtures/complex.json")
data = json.loads(fixture_path.read_text(encoding="utf-8"))

print(f"Loaded JSON with {len(data)} top-level keys:")
for key in data:
    print(f"  • {key}")

## 3. Render to HTML

The HTML renderer produces a **self-contained document** with:
- Inline CSS (dark theme with premium aesthetics)
- XSS-safe output (script tags stripped, entities escaped)
- Dollar signs escaped to `&#36;`
- Embedded markdown converted to safe HTML

In [ ]:
html_output = render_json_to_html(data, title="Complex JSON — HTML Render")

print(f"Generated HTML: {len(html_output):,} characters")
print(f"First 200 chars:\n{html_output[:200]}...")

In [ ]:
# Display the rendered HTML inline
display(HTML(html_output))

## 4. Render to Markdown

The Markdown renderer produces clean, readable output with:
- Bold keys with special characters escaped
- Embedded markdown passed **as-is** (no double-processing)
- Dollar signs escaped to `\$`
- Nested structures as indented lists

In [ ]:
md_output = render_json_to_markdown(data, title="Complex JSON — Markdown Render")

print(f"Generated Markdown: {len(md_output):,} characters")
print("---")
print(md_output[:600])
print("...")

In [ ]:
# Display the rendered Markdown inline
display(Markdown(md_output))

## 5. Edge-Case Spotlight

Let's verify specific edge cases work correctly.

In [ ]:
# ── XSS Protection ──────────────────────────────────────
xss_json = {"payload": "<script>alert('xss')</script><b>bold</b>"}

html_xss = render_json_to_html(xss_json, title="XSS Test")
assert "<script>" not in html_xss, "XSS: script tag leaked!"
assert "alert(" not in html_xss, "XSS: alert() leaked!"
print("✅ XSS payloads neutralised")

# ── Dollar Signs ─────────────────────────────────────────
dollar_json = {"price": "$100", "latex": "$$\\sum x_i$$"}

html_dollar = render_json_to_html(dollar_json)
assert "&#36;" in html_dollar, "Dollar sign not escaped in HTML!"
print("✅ Dollar signs escaped in HTML")

md_dollar = render_json_to_markdown(dollar_json)
assert "\\$" in md_dollar, "Dollar sign not escaped in Markdown!"
print("✅ Dollar signs escaped in Markdown")

# ── Unicode ──────────────────────────────────────────────
unicode_json = {"emoji": "🌍🎉🏊", "cjk": "日本語", "rtl": "مرحبا"}

html_uni = render_json_to_html(unicode_json)
for char in ["🌍", "🎉", "🏊", "日本語", "مرحبا"]:
    assert char in html_uni, f"Unicode char {char} lost in HTML!"
print("✅ All unicode preserved in HTML")

md_uni = render_json_to_markdown(unicode_json)
for char in ["🌍", "🎉", "🏊", "日本語", "مرحبا"]:
    assert char in md_uni, f"Unicode char {char} lost in Markdown!"
print("✅ All unicode preserved in Markdown")

print("\n🎉 All edge-case checks passed!")

## 6. Save Outputs to Files

In [ ]:
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

# Save HTML
html_path = output_dir / "complex_output.html"
html_path.write_text(html_output, encoding="utf-8")
print(f"📄 HTML saved to: {html_path.resolve()}")

# Save Markdown
md_path = output_dir / "complex_output.md"
md_path.write_text(md_output, encoding="utf-8")
print(f"📄 Markdown saved to: {md_path.resolve()}")

print("\n✅ Done! Open the HTML file in your browser to see the rendered result.")